# Favorite World — Fine-Tune AI4Bharat Indic-Parler-TTS (Channel Voice + 150+ Real Hindi Speakers)

**100% Free Cloud Training on Google Colab T4 GPU — zero load on your PC.**

This notebook trains **AI4Bharat Indic-Parler-TTS** (the exact engine that generated your expressive Hindi sample) on:
1. **All your Favorite World channel voiceovers** (`favorite_world_voice_dataset.zip`) tagged as `"FavoriteWorld speaks clearly in a warm, expressive, studio-quality Hindi documentary narrator voice..."`
2. **150+ real Hindi speaker recordings** streamed automatically inside Colab from **`google/fleurs` (`hi_in`)** so the model preserves rich, natural human Hindi prosody and pronunciation across speakers.

### One-Time Prerequisites:
1. Select **Runtime → Change runtime type → T4 GPU**.
2. Accept the free instant model access on Hugging Face: https://huggingface.co/ai4bharat/indic-parler-tts (`gated: auto`) and paste a free Read token (`hf_...`) in Cell 1.
3. Upload `out/favorite_world_voice_dataset.zip` to Google Drive at `MyDrive/FavoriteWorld_Voice_Model/favorite_world_voice_dataset.zip` (or upload when Cell 2 asks).
4. After training finishes once, run **Cell 1 + Cell 4** anytime to launch your `gradio.live` API in ~20 seconds and paste it into **Voiceover Studio Pro → Custom Colab / Gradio URL**.

In [ ]:
# CELL 1: Login to Hugging Face, Download Dataset & Install Parler-TTS
import os
from huggingface_hub import login
HF_TOKEN = ""  # Paste your free hf_... Read token here (or enter when prompted)
login(token=HF_TOKEN or None)

!nvidia-smi -L
!pip install -q gdown
!gdown --fuzzy "https://drive.google.com/file/d/1URMspb8xOzQ834LYQ9x5EHmqMMRzC8zs/view?usp=sharing" -O /content/favorite_world_voice_dataset.zip
!git clone -q https://github.com/huggingface/parler-tts.git /content/parler-tts || true
!pip install -q -e "/content/parler-tts[train]" "datasets==3.6.0" soundfile librosa accelerate gradio


In [ ]:
# CELL 2: Build Combined Training Dataset (Your Channel Voiceovers + 150+ Real Hindi Speakers from Google FLEURS)
import os, json, zipfile
from datasets import Dataset, DatasetDict, Audio, load_dataset, concatenate_datasets

DRIVE_ZIP = "/content/favorite_world_voice_dataset.zip"
if not os.path.exists(DRIVE_ZIP) and os.path.exists("/content/drive/MyDrive/FavoriteWorld_Voice_Model/favorite_world_voice_dataset.zip"):
    DRIVE_ZIP = "/content/drive/MyDrive/FavoriteWorld_Voice_Model/favorite_world_voice_dataset.zip"
FW_DIR = "/content/fw_dataset"
os.makedirs(FW_DIR, exist_ok=True)

if not os.path.exists(DRIVE_ZIP):
    from google.colab import files
    print("Upload favorite_world_voice_dataset.zip (from your clausd/out/ folder):")
    uploaded = files.upload()
    up_name = next(iter(uploaded.keys()))
    os.replace(up_name, DRIVE_ZIP)

with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
    zf.extractall(FW_DIR)

fw_rows = []
with open(os.path.join(FW_DIR, "metadata.jsonl"), encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        fw_rows.append({
            "audio": os.path.join(FW_DIR, item["file_name"]),
            "transcription": item["transcription"],
            "text_description": item["text_description"],
        })

print(f"Loaded {len(fw_rows)} Favorite World channel voiceover clips!")
fw_ds = Dataset.from_list(fw_rows).cast_column("audio", Audio(sampling_rate=44100))

# Stream 1,000 studio male Hindi clips from SPRINGLab/IndicTTS-Hindi + 400 FLEURS hi_in + 250 FLEURS en_in (Hinglish accent)
print("Loading studio Hindi + Hinglish datasets (SPRINGLab/IndicTTS-Hindi + google/fleurs hi_in & en_in)...")
indictts = load_dataset("SPRINGLab/IndicTTS-Hindi", split="train", streaming=True)
web_rows = []
for ex in indictts:
    if str(ex.get("gender", "")).lower() in ("male", "m", "0"):
        arr = ex["audio"]["array"]; sr = ex["audio"]["sampling_rate"]
        dur = len(arr) / float(sr)
        txt = (ex.get("text") or "").strip()
        if 1.5 <= dur <= 14.0 and txt:
            web_rows.append({"audio": ex["audio"], "transcription": txt, "text_description": "Vishal speaks clearly in a deep, warm, expressive studio-quality Hindi documentary narrator voice with crisp pronunciation and no background noise."})
    if len(web_rows) >= 1000:
        break

fleurs_hi = load_dataset("google/fleurs", "hi_in", split="train[:650]", trust_remote_code=True)
for ex in fleurs_hi:
    arr = ex["audio"]["array"]; sr = ex["audio"]["sampling_rate"]
    dur = len(arr) / float(sr)
    txt = (ex.get("raw_transcription") or "").strip()
    if 2.0 <= dur <= 14.0 and txt:
        g = "female" if ex.get("gender", 0) == 1 else "male"
        web_rows.append({"audio": ex["audio"], "transcription": txt, "text_description": f"A real Hindi {g} speaker speaks clearly in a natural conversational tone with human breathing and minimal background noise."})

fleurs_en = load_dataset("google/fleurs", "en_us", split="train[:200]", trust_remote_code=True)
for ex in fleurs_en:
    arr = ex["audio"]["array"]; sr = ex["audio"]["sampling_rate"]
    dur = len(arr) / float(sr)
    txt = (ex.get("raw_transcription") or "").strip()
    if 2.0 <= dur <= 14.0 and txt:
        web_rows.append({"audio": ex["audio"], "transcription": txt, "text_description": "Vishal speaks English words with a natural, crisp Indian documentary accent and clear studio pacing."})

web_ds = Dataset.from_list(web_rows).cast_column("audio", Audio(sampling_rate=44100))
print(f"Loaded {len(web_ds)} studio Hindi + Hinglish web clips!")

# Weight Favorite World / Vishal clips 3x so the trained voice strongly anchors on your channel + Vishal voice
combined = concatenate_datasets([fw_ds, fw_ds, fw_ds, web_ds]).shuffle(seed=42)
split = combined.train_test_split(test_size=0.05, seed=42)
COMBINED_DISK = "/content/fw_indic_parler_dataset"
os.makedirs(COMBINED_DISK, exist_ok=True)
split["train"].to_parquet(os.path.join(COMBINED_DISK, "train-00000-of-00001.parquet"))
split["test"].to_parquet(os.path.join(COMBINED_DISK, "validation-00000-of-00001.parquet"))
print("Saved combined training dataset (parquet):", len(split["train"]), "train /", len(split["test"]), "val")


In [ ]:
# CELL 3: Patch Parler-TTS for 15GB T4 GPU & Fine-Tune ai4bharat/indic-parler-tts
import os
OUT_MODEL_DIR = "/content/trained_indic_parler"
CACHE_DIR = "/content/parler_cache"
TMP_DIR = "/content/parler_tmp"
for d in (OUT_MODEL_DIR, CACHE_DIR, TMP_DIR):
    os.makedirs(d, exist_ok=True)

# Patch parler-tts training scripts for Colab Python 3.12+ & 15GB T4 VRAM (Adafactor)
dp = "/content/parler-tts/training/data.py"
ds = open(dp, encoding="utf-8").read()
ds = ds.replace(
    "metadata_dataset_names = metadata_dataset_names.split(\"+\") if metadata_dataset_names is not None else None",
    "metadata_dataset_names = metadata_dataset_names.split(\"+\") if metadata_dataset_names is not None else [None] * len(dataset_names)",
)
open(dp, "w", encoding="utf-8").write(ds)

tp = "/content/parler-tts/training/run_parler_tts_training.py"
ts = open(tp, encoding="utf-8").read()
ts = ts.replace("if bandwidth is not None:", 'if "bandwidth" in encoder_signature and bandwidth is not None:')
ts = ts.replace("del generate_labels, all_lens", "pass")
ts = ts.replace('len(vectorized_datasets["eval"]) - 1', 'len(vectorized_datasets["train"]) - 1')
ts = ts.replace("optimizer = torch.optim.AdamW(", "optimizer = transformers.Adafactor(")
ts = ts.replace("betas=(training_args.adam_beta1, training_args.adam_beta2),", "scale_parameter=False, relative_step=False, warmup_init=False,")
ts = ts.replace("eps=training_args.adam_epsilon,", "")
open(tp, "w", encoding="utf-8").write(ts)

!USE_TF=0 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True accelerate launch /content/parler-tts/training/run_parler_tts_training.py \
  --model_name_or_path "ai4bharat/indic-parler-tts" \
  --feature_extractor_name "ai4bharat/indic-parler-tts" \
  --description_tokenizer_name "ai4bharat/indic-parler-tts" \
  --prompt_tokenizer_name "ai4bharat/indic-parler-tts" \
  --train_dataset_name "/content/fw_indic_parler_dataset" \
  --train_dataset_config_name "default" \
  --train_split_name "train" \
  --eval_dataset_name "/content/fw_indic_parler_dataset" \
  --eval_dataset_config_name "default" \
  --eval_split_name "validation" \
  --target_audio_column_name "audio" \
  --description_column_name "text_description" \
  --prompt_column_name "transcription" \
  --save_to_disk "{CACHE_DIR}" \
  --temporary_save_to_disk "{TMP_DIR}" \
  --output_dir "{OUT_MODEL_DIR}" \
  --overwrite_output_dir \
  --do_train \
  --fp16 \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 8 \
  --gradient_checkpointing \
  --learning_rate 2e-5 \
  --num_train_epochs 3 \
  --warmup_steps 30 \
  --logging_steps 10 \
  --save_steps 200 \
  --save_total_limit 2 \
  --report_to "none" \
  --preprocessing_num_workers 2


In [ ]:
# CELL 4: Serve Your Trained Favorite World Indic-Parler-TTS Voice (Works with Voiceover Studio Pro!)
# Run Cell 1 + this cell anytime after training — loads from Google Drive in ~20 seconds.
import os, tempfile, torch, soundfile as sf, gradio as gr
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

MODEL_DIR = "/content/trained_indic_parler" if os.path.exists("/content/trained_indic_parler/config.json") else "/content/drive/MyDrive/FavoriteWorld_Voice_Model/trained_indic_parler"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

model = ParlerTTSForConditionalGeneration.from_pretrained(MODEL_DIR).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
desc_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)
print("Loaded Trained Favorite World Indic-Parler-TTS!")

DEFAULT_DESC = (
    "FavoriteWorld speaks clearly in a warm, expressive, studio-quality Hindi documentary "
    "narrator voice with natural pacing and no background noise."
)

def generate_finetuned(text_input, description_input):
    desc = (description_input or "").strip() or DEFAULT_DESC
    if "FavoriteWorld" not in desc and any(k in desc for k in ("Rohit", "Aman", "Divya", "Rani")):
        desc = DEFAULT_DESC
    desc_ids = desc_tokenizer(desc, return_tensors="pt").to(device)
    prompt_ids = tokenizer(text_input, return_tensors="pt").to(device)
    with torch.no_grad():
        generation = model.generate(
            input_ids=desc_ids.input_ids,
            attention_mask=desc_ids.attention_mask,
            prompt_input_ids=prompt_ids.input_ids,
            prompt_attention_mask=prompt_ids.attention_mask,
        )
    audio_arr = generation.cpu().numpy().squeeze()
    out_wav = tempfile.NamedTemporaryFile(suffix=".wav", delete=False).name
    sf.write(out_wav, audio_arr, model.config.sampling_rate)
    return out_wav

with gr.Blocks() as demo:
    gr.Markdown("# Favorite World — Trained Indic-Parler-TTS Cloud Voice")
    txt = gr.Textbox(label="Input Text")
    desc = gr.Textbox(value=DEFAULT_DESC, label="Description")
    btn = gr.Button("Generate Audio")
    out = gr.Audio(type="filepath", label="Generated Audio")
    btn.click(generate_finetuned, [txt, desc], [out], api_name="generate_finetuned")

demo.queue().launch(share=True, debug=True)
